# 04 - Labels prüfen und Datenaufteilung festlegen
Die kuratierten Bilder werden den 248 TLFS23-Klassen zugeordnet und anhand der veröffentlichten Ordner-Zeichen-Zuordnung geprüft. Anschließend wird der Datenbestand reproduzierbar in Training, Validation und Test aufgeteilt. Bereits verwendete Bilder sowie die in Notebook 03 gebildeten dHash-Gruppen werden berücksichtigt. Der Testsplit wird nicht zur visuellen Prüfung verwendet.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

here = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (here, *here.parents)
                     if (p / "notebooks").is_dir() and (p / "data").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Projektordner mit notebooks/ und data/ fehlt.")
for name in ["data.py", "visual_review.py"]:
    if not (PROJECT_ROOT / "src/tlfs23" / name).is_file():
        raise FileNotFoundError(f"src/tlfs23/{name} fehlt. Dateien aus dem Paket ergänzen.")
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from tlfs23 import data as d, visual_review as r
from tlfs23.common import paths, read_csv

P = paths(PROJECT_ROOT)
print("Projekt:", PROJECT_ROOT)

candidates = d.load_curated(PROJECT_ROOT)
print("Kandidaten:", len(candidates))
print("Ordner:", candidates["dataset_folder"].nunique())

## 1. Ordner-Labels zuordnen
Vorhandene Einträge bleiben erhalten. Korrekturen werden in `FOLDER_TO_CHARACTER` eingetragen
und müssen erneut bestätigt werden. Die Referenz ist [TLFS23, Tabelle 1](https://pmc.ncbi.nlm.nih.gov/articles/PMC10790027/).
Nummerierte Ordner werden nicht automatisch mit Schemaindizes gleichgesetzt.

In [ ]:
review = d.label_review(PROJECT_ROOT, candidates)

FOLDER_TO_CHARACTER = {}  # Nur anhand der Referenz geprüfte Korrekturen.
review = r.apply_mapping_updates(
    PROJECT_ROOT,
    review,
    FOLDER_TO_CHARACTER,
)

view = r.label_view(review)

display(
    view.groupby("category")
        .size()
        .rename("n_classes")
        .to_frame()
)

print(
    "Fehlende/unbekannte Zuordnungen:",
    int(view["category"].isna().sum())
)

print("Prüftabelle:", P["config"] / "label_review.csv")

## 2. Zuordnung an Bildern prüfen
Jedes Bild zeigt Ordner, eingetragenes Zeichen, Kategorie und Komponenten. Nicht zugeordnete Labels
bleiben sichtbar. Die Galerie bestätigt nichts automatisch und ersetzt nicht den Quellenabgleich aller Labels.
Vor dem Split werden diese Bilder als verwendet protokolliert; nach dem Split zeigt die Galerie nur Trainingsbilder.

In [ ]:
REVIEW_FOLDERS = (
    view.dropna(subset=["category"])
        .groupby("category", sort=False)
        .head(1)["dataset_folder"]
        .tolist()
)

mapping_examples = r.folder_examples(
    PROJECT_ROOT,
    candidates,
    view,
    folders=REVIEW_FOLDERS,
    per_folder=2,
)

r.show_cards(
    PROJECT_ROOT,
    mapping_examples,
    "Ordnerzuordnung prüfen",
    context="04_mapping",
    output_name="04_mapping_examples",
)

## 3. Mapping bestätigen
`True` bestätigt den Quellenabgleich aller verwendeten Ordnerlabels, nicht nur der gezeigten Beispiele.
Bei einer falschen Zuordnung zuerst den Eintrag korrigieren. Bestehende Belege werden nicht pauschal ersetzt.

In [ ]:
CONFIRM_FOLDER_MAPPING = True
MAPPING_EVIDENCE = (
    "Alle 248 Dataset-Ordner wurden mit der offiziellen TLFS23-"
    "Ordner-zu-Zeichen-Zuordnung abgeglichen. Zusätzlich wurden "
    "Beispielbilder der Ordner visuell auf Plausibilität geprüft."
)

mapping_ready = False
if CONFIRM_FOLDER_MAPPING:
    if mapping_examples.empty:
        raise ValueError("Zuerst eine Bildstichprobe der Ordnerzuordnung anzeigen.")
    review = read_csv(P["config"] / "label_review.csv")
    schema = r.approve_mapping(PROJECT_ROOT, review, MAPPING_EVIDENCE)
    mapping_ready = True
    view = r.label_view(read_csv(P["config"] / "label_review.csv"))
    display(schema.groupby("category").size().rename("n_classes").to_frame())
else:
    print("Mapping noch nicht freigegeben. Galerie und Referenz prüfen.")

## 4. Datenhistorie prüfen und Split festschreiben
Historische Manifeste und die bisherigen Sichtproben sind für Validation und Holdout gesperrt.
`EXPOSURE_REVIEWED` bestätigt, dass die Datenhistorie vollständig angegeben wurde.
Ohne verifizierte Personen-/Session-IDs bleibt die Aufteilung bildbezogen. Ein vorhandener Split wird nicht neu gezogen.

In [ ]:
SPLIT_CONFIG = {
    "seed": 42,
    "train": 180,
    "focus_train": 360,
    "validation": 50,
    "test": 50,
    "background_eval": 100,
}

# Bereits früher verwendete Bilder
EXPOSURE_FILES = [
    "data/manifests/labelled_split_manifest.csv"
]

EXPOSURE_REVIEWED = True
NO_PREVIOUS_EXPOSURE = False

# dHash-Gruppen aus Notebook 03
GROUPS_FILE = P["manifests"] / "split_group_manifest.csv"

if not GROUPS_FILE.exists():
    raise FileNotFoundError(
        f"Split-Gruppen aus Notebook 03 fehlen: {GROUPS_FILE}"
    )

SPLIT_NOTE = (
    "Das frühere labelled_split_manifest enthält 29.760 bereits verwendete "
    "Bilder und wurde vollständig mit dem aktuellen kuratierten Datenbestand "
    "abgeglichen. Diese Bilder werden als frühere Datenexposition berücksichtigt. "
    "Zusätzlich werden gleiche dHash-Gruppen aus Notebook 03 beim Split "
    "zusammengehalten, um sehr ähnliche Aufnahmen nicht über mehrere Splits "
    "zu verteilen."
)

plan = None

if mapping_ready and EXPOSURE_REVIEWED:
    plan, split_info = r.freeze_or_resume(
        PROJECT_ROOT,
        SPLIT_CONFIG,
        EXPOSURE_FILES,
        EXPOSURE_REVIEWED,
        SPLIT_NOTE,
        groups_file=GROUPS_FILE.relative_to(PROJECT_ROOT).as_posix(),
        no_previous_exposure=NO_PREVIOUS_EXPOSURE,
    )

    display(pd.crosstab(plan["split"], plan["category"]))
    print(
    "Limitation: Es liegen keine verifizierten Personen- oder Session-IDs vor. "
    "Die Gruppentrennung basiert auf den dHash-Gruppen aus Notebook 03."
    )
else:
    print(
        "Noch kein Split-Aufruf: "
        "Mapping und Datenhistorie zuerst bestätigen."
    )

In [ ]:
exposure_path = PROJECT_ROOT / "data/manifests/labelled_split_manifest.csv"
old_split = read_csv(exposure_path)

old_paths = set(old_split["relative_path"])
current_paths = set(candidates["relative_path"])
overlap = old_paths & current_paths

print("Historisch verwendete Bilder:", len(old_paths))
print("Davon im aktuellen Bestand:", len(overlap))

if len(overlap) != len(old_paths):
    raise ValueError(
        "Historische Datenexposition ist nicht vollständig nachvollziehbar."
    )

## 5. Handstruktur auf Trainingsbildern prüfen
Die Handstruktur wird ausschließlich anhand von Trainingsbildern geprüft. Dabei werden Uyir, Mei, Uyirmei, Ayudha und Background betrachtet. Die dargestellten Komponenten stammen aus dem bestätigten Labelschema und nicht aus MediaPipe-Vorhersagen.

In [ ]:
HAND_IMAGES_PER_CATEGORY = 3
hand_examples = pd.DataFrame()
if plan is not None:
    hand_examples = r.hand_examples(plan, view, per_category=HAND_IMAGES_PER_CATEGORY)
    for category in r.CATEGORIES:
        examples = hand_examples.loc[hand_examples["category"].eq(category)]
        r.show_cards(PROJECT_ROOT, examples, "Handstruktur: " + category,
                     context="04_hand", output_name="04_hand_" + category)
else:
    print("Handprüfung folgt nach dem festgeschriebenen Split.")

## 6. Handstruktur bestätigen
Die Notiz beschreibt die tatsächlichen Beobachtungen zu Uyir, Mei, Uyir-Mei, Ayudha und Background
sowie offene Fälle. `FIST` ist eine sichtbare Faust, `NOT_DETECTED` ein späteres Detektionsergebnis.
Die Freigabe wird zusammen mit den gezeigten Beispielen dokumentiert.

In [ ]:
CONFIRM_HAND_STRUCTURE = True
HAND_REVIEW_NOTE = ("Die Stichprobe zeigt unterschiedliche Zwei-Hand-Konfigurationen für Uyir, Mei "
    "und Uyirmei. FIST bezeichnet dabei eine sichtbare Faust und keinen "
    "Detektionsfehler. Ayudha wird als eigene Kategorie behandelt. "
    "Background enthält keine festgelegte gültige Tamil-Zeichenkonfiguration." )

hand_ready = False
if CONFIRM_HAND_STRUCTURE and plan is not None:
    r.approve_hands(PROJECT_ROOT, hand_examples, HAND_REVIEW_NOTE)
    hand_ready = True
    print("Handstruktur für diesen Split dokumentiert.")
else:
    print("Handstruktur noch nicht freigegeben.")

In [ ]:
checked_plan, _ = d.load_split(PROJECT_ROOT)

print("Freigegebene Bilder:", len(checked_plan))
print("Split:", P["manifests"] / "split_manifest.csv")

## Ergebnis 
Alle 248 TLFS23-Klassen wurden einem bestätigten Zeichen und einer Kategorie zugeordnet. Der festgeschriebene Datensatz umfasst 50.400 Trainingsbilder sowie jeweils 12.450 Bilder für Validation und Test. Historisch verwendete Bilder werden bei der Aufteilung berücksichtigt; ähnliche Bilder aus denselben dHash-Gruppen werden nicht über mehrere Splits verteilt. Eine Unabhängigkeit nach Personen oder Aufnahmesessions kann mangels entsprechender verifizierter Metadaten nicht garantiert werden. Die Handstruktur wurde ausschließlich anhand des Trainingssplits geprüft und dokumentiert.